# ViFinQA — Qwen3-4B-Instruct-2507 sinh `ProgramDecision` masked-PAL (Kaggle T4)

Notebook này là **bước 2** của đường masked-PAL (spec 2026-08-24). Bước 1
(`notebooks/kaggle_program_batches_t4.ipynb`, lệnh `submission row-batches`)
đã sinh payload `batch_*.jsonl`: mỗi câu hỏi kèm danh sách ứng viên Ô **đã đánh
số nhưng không có giá trị**. Ở đây `Qwen/Qwen3-4B-Instruct-2507` (Apache-2.0,
biến thể **non-thinking**, <14B theo ràng buộc dự án) chọn ô và viết chương
trình số học cho từng câu, xuất ra `data/decisions/program-full.jsonl`.

**Không cài repo package trên Kaggle** (`requires-python <3.12` xung đột kernel
Python 3.12): notebook đọc thẳng payload JSON và chỉ làm *light validation*
(parse JSON + đủ khóa + cấm chữ số trong `program` bằng regex + kiểm range).
**Strict validation chạy local**, sau khi tải kết quả về:

```bash
uv run python scripts/validate_program_decisions.py \
    --batches-dir artifacts/batches/program-full \
    --decisions data/decisions/program-full.jsonl
```

## Điều kiện cần

- **Accelerator: GPU T4** (1 GPU đủ — model 4B fp16 ~8GB trên 16GB VRAM).
- **Internet: On** (tải model ~8GB từ Hugging Face).
- Payload batches đã upload thành dataset Kaggle tên `vifinqa-program-payload`
  (giữ cấu trúc `artifacts/batches/program-full/batch_*.jsonl`) **hoặc** chép
  vào `/kaggle/working/artifacts/batches/program-full/`.

## Thời gian dự kiến

~1012 prompts (trong đó ~953 câu có ứng viên; 59 câu 0 ứng viên bị loại sẵn).
Tổng 30–60 phút kể cả `pip install vllm` (~5 phút) và tải model (~8GB).

## Luồng các cell

| Cell | Nội dung |
| --- | --- |
| 2 | Kiểm môi trường + cài vLLM |
| 3 | Đọc payload, dựng prompt (ba quy tắc cứng + hai few-shot tự biên soạn) |
| 4 | Sinh offline bằng vLLM (`temperature=0`) + parse + light-validate |
| 5 | Retry đúng MỘT lần cho câu lỗi (ghim thông báo lỗi vào prompt) |
| 6 | Ghi `data/decisions/program-full.jsonl` + `failures.jsonl` + tổng kết |
| 7 | Đóng gói `program-decisions.tar.gz` + FileLink tải về |

Câu vẫn fail sau retry được ghi nguyên trạng vào `failures.jsonl` kèm lý do —
**không bao giờ bịa quyết định thay thế**.


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                      "--format=csv"], capture_output=True, text=True).stdout)

work_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
free_gb = shutil.disk_usage(work_dir).free / 1e9
print(f"dia lam viec: {work_dir} (con trong {free_gb:.1f} GB)")
assert free_gb > 20, "Can >20GB trong cho model + output"

# Cai vLLM vao dung environment cua kernel (Python 3.12 cua Kaggle).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "vllm"], check=True)

import torch
import vllm

print("vllm:", vllm.__version__, "| torch:", torch.__version__,
      "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Khong thay GPU -- kiem lai Accelerator cua session"


In [ ]:
import json
import os
import re
from pathlib import Path


def _discover_payload_dir() -> Path:
    override = os.environ.get("VIFINQA_PAYLOAD_DIR")  # cho phep chay thu local
    candidates = ([Path(override)] if override else []) + [
        Path("/kaggle/input/vifinqa-program-payload/artifacts/batches/program-full"),
        Path("/kaggle/input/vifinqa-program-payload"),
        Path("/kaggle/working/artifacts/batches/program-full"),
    ]
    for candidate in candidates:
        if candidate.is_dir() and any(candidate.glob("batch_*.jsonl")):
            return candidate
    searched = ", ".join(str(item) for item in candidates)
    raise FileNotFoundError("Khong thay batch_*.jsonl trong bat ky thu muc nao: " + searched)


PAYLOAD_DIR = _discover_payload_dir()

payloads: list[dict] = []
for batch_path in sorted(PAYLOAD_DIR.glob("batch_*.jsonl")):
    for line in batch_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            payloads.append(json.loads(line))

question_ids = [item["question_id"] for item in payloads]
assert len(question_ids) == len(set(question_ids)), "trung question_id trong payload"
print(f"{PAYLOAD_DIR}: {len(payloads)} cau")

# Cau 0 ung vien khong the co ProgramDecision hop le (cells phai >= 1 phan tu):
# loai ngay tu dau, khong gui qua model, se ghi vao failures.jsonl o cell 6.
generatable = [item for item in payloads if item["candidates"]]
no_candidate = [item for item in payloads if not item["candidates"]]
print(f"{len(generatable)} cau co ung vien, {len(no_candidate)} cau 0 ung vien")

# ---------------------------------------------------------------------------
# Khối hướng dẫn hệ thống (tiếng Việt) ghim đúng hợp đồng: ProgramDecision
# (execution/program_contracts.py) + AST guard (execution/masked_program.py).
# ---------------------------------------------------------------------------
SYSTEM_INSTRUCTION = """Bạn là bộ sinh quyết định cho hệ thống hỏi đáp báo cáo tài chính Việt Nam theo kiến trúc masked-PAL: bạn KHÔNG bao giờ thấy giá trị số của ô dữ liệu; bạn chỉ chọn ô và viết chương trình số học, hệ thống tự tính đáp án từ CSV.

Mỗi câu hỏi kèm danh sách ứng viên Ô đã đánh số, mỗi ứng viên có các trường: index, company_code, row_path, col_path, period, statement_type, unit.

Ba quy tắc BẮT BUỘC (vi phạm là quyết định bị loại):
1. `program` chỉ được chứa placeholder dạng `[NUM_i]`, các toán tử `+`, `-`, `*`, `/`, dấu ngoặc `( )` và hàm `abs(...)`. CẤM MỌI CHỮ SỐ và mọi hằng số khác trong `program`. Muốn đổi thang đơn vị, dùng trường `scale`.
2. `scale` nhận ĐÚNG MỘT giá trị trong enum: "none", "percent", "thousand", "million", "billion". Nghĩa áp thang lên kết quả: "none" giữ nguyên; "percent" nhân 100; "thousand", "million", "billion" lần lượt chia 1000, 1000000, 1000000000. Chọn theo đơn vị mà câu hỏi yêu cầu so với `unit` của các ô được dùng.
3. Mỗi phần tử của mảng `cells` là một vị trí (đúng giá trị trường `index`) trong DANH SÁCH ỨNG VIÊN CỦA CHÍNH CÂU ĐÓ. Placeholder đánh số theo THỨ TỰ PHẦN TỬ của `cells`: `[NUM_0]` ứng với phần tử ĐẦU TIÊN của `cells`, `[NUM_1]` là phần tử THỨ HAI, … KHÔNG phải giá trị `index` của ứng viên.

Trường `uses` chống trượt chỉ số: mỗi vị trí i trong `cells` cần đúng một phần tử dạng {"num": i, "row": "<tên dòng>", "col": "<tên cột/kỳ>"} mô tả [NUM_i] thực sự là ô nào. `row` lấy nguyên văn (hoặc phần đuôi) của `row_path`; `col` nêu rõ kỳ/năm của ô.

Chỉ trả về DUY NHẤT một đối tượng JSON với đúng các trường: question_id (nguyên số), cells (mảng số nguyên), program (chuỗi), uses (mảng đối tượng num/row/col), scale (chuỗi enum). Không giải thích, không bọc Markdown."""


# Hai few-shot TỰ BIÊN SOẠN (không lấy từ payload thật): (A) tra trực tiếp 1 ô,
# (B) phép tính 2 ô kèm đổi thang percent.
FEWSHOT_A = {
    "question_id": 900001,
    "question": "Doanh thu thuần dịch vụ chứng khoán năm 2023 của CTCP Chứng khoán TCBS là bao nhiêu tỷ đồng?",
    "companies": ["TCBS"],
    "periods": ["2023"],
    "candidates": [
        {"index": 0, "company_code": "TCBS", "row_path": "Doanh thu thuần dịch vụ chứng khoán", "col_path": "Năm 2023", "period": 2023, "statement_type": None, "unit": "tỷ đồng"},
        {"index": 1, "company_code": "TCBS", "row_path": "Doanh thu thuần dịch vụ chứng khoán", "col_path": "Năm 2022", "period": 2022, "statement_type": None, "unit": "tỷ đồng"},
        {"index": 2, "company_code": "TCBS", "row_path": "Chi phí lãi vay", "col_path": "Năm 2023", "period": 2023, "statement_type": None, "unit": "tỷ đồng"},
    ],
}
FEWSHOT_A_DECISION = {
    "question_id": 900001,
    "cells": [0],
    "program": "[NUM_0]",
    "uses": [{"num": 0, "row": "Doanh thu thuần dịch vụ chứng khoán", "col": "Năm 2023"}],
    "scale": "none",
}

FEWSHOT_B = {
    "question_id": 900002,
    "question": "Tốc độ tăng trưởng phần trăm (%) doanh thu hoạt động ngân hàng năm 2023 so với năm 2022 của CTCP Chứng khoán TCBS là bao nhiêu?",
    "companies": ["TCBS"],
    "periods": ["2023", "2022"],
    "candidates": [
        {"index": 0, "company_code": "TCBS", "row_path": "Lợi nhuận trước thuế", "col_path": "Năm 2023", "period": 2023, "statement_type": None, "unit": "tỷ đồng"},
        {"index": 1, "company_code": "TCBS", "row_path": "Lợi nhuận trước thuế", "col_path": "Năm 2022", "period": 2022, "statement_type": None, "unit": "tỷ đồng"},
        {"index": 2, "company_code": "TCBS", "row_path": "Chi phí hoạt động", "col_path": "Năm 2023", "period": 2023, "statement_type": None, "unit": "tỷ đồng"},
        {"index": 3, "company_code": "TCBS", "row_path": "Doanh thu hoạt động ngân hàng", "col_path": "Năm 2023", "period": 2023, "statement_type": None, "unit": "tỷ đồng"},
        {"index": 4, "company_code": "TCBS", "row_path": "Chi phí hoạt động", "col_path": "Năm 2022", "period": 2022, "statement_type": None, "unit": "tỷ đồng"},
        {"index": 5, "company_code": "TCBS", "row_path": "Doanh thu hoạt động ngân hàng", "col_path": "Năm 2022", "period": 2022, "statement_type": None, "unit": "tỷ đồng"},
    ],
}
FEWSHOT_B_DECISION = {
    # cells[0] là ứng viên index 3, cells[1] là ứng viên index 5 ->
    # [NUM_0]/[NUM_1] đánh số theo THỨ TỰ này, không phải theo index.
    "question_id": 900002,
    "cells": [3, 5],
    "program": "([NUM_0] - [NUM_1]) / [NUM_1]",
    "uses": [
        {"num": 0, "row": "Doanh thu hoạt động ngân hàng", "col": "Năm 2023"},
        {"num": 1, "row": "Doanh thu hoạt động ngân hàng", "col": "Năm 2022"},
    ],
    "scale": "percent",
}


def render_user_content(payload: dict) -> str:
    """Một lượt user: câu hỏi + toàn bộ ứng viên, KHÔNG chứa giá trị số nào."""
    return (
        f"câu hỏi: {payload['question']}\n"
        f"companies: {json.dumps(payload['companies'], ensure_ascii=False)}\n"
        f"periods: {json.dumps(payload['periods'], ensure_ascii=False)}\n"
        f"ứng viên Ô: {json.dumps(payload['candidates'], ensure_ascii=False)}"
    )


FEWSHOT_MESSAGES = [
    {"role": "user", "content": render_user_content(FEWSHOT_A)},
    {"role": "assistant", "content": json.dumps(FEWSHOT_A_DECISION, ensure_ascii=False)},
    {"role": "user", "content": render_user_content(FEWSHOT_B)},
    {"role": "assistant", "content": json.dumps(FEWSHOT_B_DECISION, ensure_ascii=False)},
]


def build_messages(payload: dict) -> list:
    messages = [{"role": "system", "content": SYSTEM_INSTRUCTION}]
    messages.extend(FEWSHOT_MESSAGES)
    messages.append({"role": "user", "content": render_user_content(payload)})
    return messages


# ---------------------------------------------------------------------------
# Light validation — CHỈ để sàng trên Kaggle, không thay thế strict validator
# local (scripts/validate_program_decisions.py).
# ---------------------------------------------------------------------------
SCALE_VALUES = ("none", "percent", "thousand", "million", "billion")
DECISION_KEYS = {"question_id", "cells", "program", "uses", "scale"}
_NUM_TOKEN = re.compile(r"\[NUM_(\d+)\]")


def extract_json_object(text):
    """Tách khối JSON ngoài cùng (bỏ qua code fence / lời dẫn nếu model bọc)."""
    start = text.find("{")
    if start < 0:
        return None, "json_unparseable:khong_co_khoi_JSON"
    depth = 0
    in_string = False
    escaped = False
    for position in range(start, len(text)):
        character = text[position]
        if in_string:
            if escaped:
                escaped = False
            elif character == "\\":
                escaped = True
            elif character == '"':
                in_string = False
            continue
        if character == '"':
            in_string = True
        elif character == "{":
            depth += 1
        elif character == "}":
            depth -= 1
            if depth == 0:
                try:
                    obj = json.loads(text[start : position + 1])
                except json.JSONDecodeError as error:
                    return None, f"json_unparseable:{error.msg}"
                if not isinstance(obj, dict):
                    return None, "json_unparseable:khong_phai_object"
                return obj, None
    return None, "json_unparseable:khoi_JSON_khong_dong_day_du"


def validate_decision(obj, payload):
    """Kiểm nhẹ một quyết định; trả về danh sách lý do lỗi (rỗng = đạt).

    Ghim đúng ba quy tắc cứng: đủ khóa, không chữ số/ký tự lạ trong `program`,
    `scale` thuộc enum, `cells`/`[NUM_i]` trong phạm vi, `uses` phủ đủ vị trí.
    """
    errors = []
    keys = set(obj)
    missing = sorted(DECISION_KEYS - keys)
    extra = sorted(keys - DECISION_KEYS)
    if missing:
        errors.append("missing_keys:" + ",".join(missing))
    if extra:
        errors.append("extra_keys:" + ",".join(extra))
    if missing or extra:
        return errors
    if obj["question_id"] != payload["question_id"]:
        errors.append(f"question_id_mismatch:{obj['question_id']!r}")

    cells = obj["cells"]
    n_candidates = len(payload["candidates"])
    if not isinstance(cells, list) or not cells or not all(isinstance(c, int) for c in cells):
        errors.append("cells_bad_shape")
        return errors
    out_of_range = [c for c in cells if c < 0 or c >= n_candidates]
    if out_of_range:
        errors.append(f"cell_index_out_of_range:{out_of_range}")

    program = obj["program"]
    if not isinstance(program, str) or not program.strip():
        errors.append("program_not_string")
        return errors
    without_nums = _NUM_TOKEN.sub("", program)
    if re.search(r"\d", without_nums):
        errors.append("digit_in_program")
    stray = re.search(r"[^+\-*/()\sab]", without_nums)
    if stray:
        errors.append(f"bad_char_in_program:{stray.group(0)!r}")
    overflow = sorted({int(m) for m in _NUM_TOKEN.findall(program) if int(m) >= len(cells)})
    if overflow:
        errors.append(f"num_index_out_of_range:{overflow}")

    scale = obj["scale"]
    if scale not in SCALE_VALUES:
        errors.append(f"scale_not_in_enum:{scale!r}")

    uses = obj["uses"]
    if not isinstance(uses, list) or not all(isinstance(u, dict) for u in uses):
        errors.append("uses_bad_shape")
        return errors
    if any(set(u) != {"num", "row", "col"} or not isinstance(u.get("row"), str)
           or not u["row"].strip() or not isinstance(u.get("col"), str) for u in uses):
        errors.append("uses_entry_bad_shape")
    elif sorted(u["num"] for u in uses if isinstance(u.get("num"), int)) != list(range(len(cells))):
        errors.append("uses_not_covering_cells")
    return errors


def canonical_decision(payload, obj):
    """Chuẩn hóa đúng thứ tự trường như contract ProgramDecision."""
    return {
        "question_id": payload["question_id"],
        "cells": list(obj["cells"]),
        "program": obj["program"],
        "uses": [{"num": u["num"], "row": u["row"], "col": u["col"]} for u in obj["uses"]],
        "scale": obj["scale"],
    }


sizes = sorted(len(render_user_content(item)) for item in generatable)
print(f"user-content chars: p50={sizes[len(sizes) // 2]} max={sizes[-1]}")
print("--- mau user-content (cat bot) ---")
print(render_user_content(generatable[0])[:600])


In [ ]:
from vllm import LLM, SamplingParams

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
# MAX_ATTEMPTS = 2: mỗi câu tối đa 2 lượt — lần 1 dưới đây và ĐÚNG MỘT lần
# sinh lại ở cell kế tiếp (retry ghim thông báo lỗi vào prompt).
MAX_ATTEMPTS = 2

llm = LLM(
    model=MODEL_ID,
    dtype="float16",  # T4 không hỗ trợ bf16; 4B fp16 ~8GB vừa 16GB VRAM
    max_model_len=8192,
    gpu_memory_utilization=0.90,
)
tokenizer = llm.get_tokenizer()


def render_prompt(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


SAMPLING = SamplingParams(temperature=0, max_tokens=512)

round1 = llm.generate([render_prompt(build_messages(item)) for item in generatable], SAMPLING)

decisions_by_qid = {}
failure_reasons = {}
raw_outputs = {}
for payload, output in zip(generatable, round1):
    text = output.outputs[0].text
    qid = payload["question_id"]
    raw_outputs[qid] = [text]
    obj, parse_error = extract_json_object(text)
    errors = [parse_error] if parse_error else validate_decision(obj, payload)
    if errors:
        failure_reasons[qid] = errors
    else:
        decisions_by_qid[qid] = canonical_decision(payload, obj)

print(f"Lan 1: {len(decisions_by_qid)} hop le, {len(failure_reasons)} loi")


In [ ]:
RETRY_INSTRUCTION_TEMPLATE = """Bạn trả lời ở lượt trước KHÔNG hợp lệ. Các lỗi phát hiện:
{errors}
Hãy sinh lại DUY NHẤT một đối tượng JSON đúng quy tắc đã nêu.
Nhắc nhanh:
- `program` CẤM mọi chữ số; chỉ `[NUM_i]`, `+`, `-`, `*`, `/`, ngoặc, `abs()`.
- `scale` chỉ nhận "none" | "percent" | "thousand" | "million" | "billion".
- `[NUM_i]` đánh số theo THỨ TỰ phần tử của `cells`, không phải giá trị `index`.
- Mỗi vị trí trong `cells` cần đúng một mục `uses` tương ứng."""

if failure_reasons:
    retry_payloads = [item for item in generatable if item["question_id"] in failure_reasons]
    retry_prompts = []
    for payload in retry_payloads:
        messages = build_messages(payload)
        messages.append({"role": "assistant",
                         "content": raw_outputs[payload["question_id"]][-1]})
        messages.append({"role": "user", "content": RETRY_INSTRUCTION_TEMPLATE.format(
            errors="\n".join("- " + reason for reason in failure_reasons[payload["question_id"]])
        )})
        retry_prompts.append(render_prompt(messages))

    retry_round = llm.generate(retry_prompts, SAMPLING)
    recovered = 0
    for payload, output in zip(retry_payloads, retry_round):
        text = output.outputs[0].text
        qid = payload["question_id"]
        raw_outputs[qid].append(text)
        obj, parse_error = extract_json_object(text)
        errors = [parse_error] if parse_error else validate_decision(obj, payload)
        if errors:
            failure_reasons[qid] = errors  # vẫn fail: cập nhật lý do mới nhất
        else:
            decisions_by_qid[qid] = canonical_decision(payload, obj)
            failure_reasons.pop(qid)
            recovered += 1
    print(f"Retry (duy nhat 1 lan): cuu duoc {recovered}/{len(retry_payloads)}, "
          f"van fail {len(failure_reasons)}")
else:
    print("Khong co cau nao can retry")


In [ ]:
import json
from collections import Counter
from pathlib import Path

OUT_DIR = Path("/kaggle/working/data/decisions")
DECISIONS_PATH = OUT_DIR / "program-full.jsonl"
FAILURES_PATH = Path("/kaggle/working/failures.jsonl")

OUT_DIR.mkdir(parents=True, exist_ok=True)

with DECISIONS_PATH.open("w", encoding="utf-8") as handle:
    # Giữ đúng thứ tự payload: quyết định ứng question_id 1:1 với payload.
    for payload in payloads:
        decision = decisions_by_qid.get(payload["question_id"])
        if decision is not None:
            handle.write(json.dumps(decision, ensure_ascii=False) + "\n")

reason_counter = Counter()
with FAILURES_PATH.open("w", encoding="utf-8") as handle:
    for payload in payloads:
        qid = payload["question_id"]
        reasons = failure_reasons.get(qid)
        if reasons is None and qid not in decisions_by_qid:
            reasons = ["no_cell_candidates"]  # 0 ứng viên: không thể có quyết định
        if not reasons:
            continue
        for reason in reasons:
            reason_counter[reason.split(":")[0]] += 1
        handle.write(json.dumps({
            "question_id": qid,
            "reasons": reasons,
            "attempts": len(raw_outputs.get(qid, [])),
            "last_raw_output": (raw_outputs.get(qid) or [""])[-1][:2000],
        }, ensure_ascii=False) + "\n")

written = sum(1 for line in DECISIONS_PATH.open(encoding="utf-8") if line.strip())
failed = sum(1 for line in FAILURES_PATH.open(encoding="utf-8") if line.strip())
scale_counter = Counter(item["scale"] for item in decisions_by_qid.values())

print(f"payload cau hoi : {len(payloads)}")
print(f"decisions       : {written} -> {DECISIONS_PATH}")
print(f"failures        : {failed} -> {FAILURES_PATH}")
print(f"hop le tong cong: {len(decisions_by_qid)}")
print(f"phan bo scale   : {dict(scale_counter)}")
print(f"ly do that bai  : {dict(reason_counter)}")
# Bất biến: mỗi payload rơi vào đúng một trong hai file.
assert written + failed == len(payloads), "thieu hoac thua cau khi ghi file"


In [ ]:
import os
import subprocess
from pathlib import Path

from IPython.display import FileLink

# FileLink chỉ hiện nút tải cho đường dẫn TƯƠNG ĐỐI tính từ cwd của kernel —
# phải chdir về /kaggle/working TRƯỚC (cạm bẫy từng làm operator mất thời gian).
os.chdir("/kaggle/working")

archive = Path("program-decisions.tar.gz")
subprocess.run([
    "tar", "czf", str(archive),
    os.path.relpath(DECISIONS_PATH, "/kaggle/working"),
    os.path.relpath(FAILURES_PATH, "/kaggle/working"),
], check=True)

print(f"{archive} ({archive.stat().st_size / 1e6:.2f} MB) — bam nut tai o output ben duoi.")
FileLink(str(archive))
